# Leer y guardar Parquet en HDFS con Spark

Spark ejecuta la suma y HDFS guarda la entrada y la salida. Completa la [guía de entorno](../01_environment_pyspark.md) y arranca el HDFS del curso.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[2]").appName("Pedidos en HDFS").getOrCreate()

pedidos = spark.createDataFrame(
    [("zona_3", 2), ("zona_1", 7), ("zona_3", 4)],
    ["zona", "unidades"],
)
total_esperado = 6

## Ejercicio · Leer, procesar y guardar en HDFS

Este ejercicio requiere HDFS en funcionamiento y que Spark pueda llegar al NameNode y a los DataNodes. Si Hadoop corre en Docker, comprueba esa conectividad desde donde se ejecuta Spark. Desde la terminal de Hadoop, consulta la dirección con `hdfs getconf -confKey fs.defaultFS` y verifica tu acceso con `hdfs dfs -ls .`.

1. Elige una carpeta nueva de HDFS donde tengas permiso de escritura. En la celda de preparación, indica su URI completa. Spark guardará allí la tabla de pedidos como entrada.
2. Vuelve a leer la entrada desde HDFS. Agrupa por zona y suma las unidades. Guarda el resumen en otra carpeta de HDFS.
3. Lee el resumen desde HDFS y comprueba que la zona 3 coincide con el total esperado de 6 unidades. Desde la terminal, ejecuta `hdfs dfs -ls URI_DEL_RESUMEN` y busca los archivos `part-*.parquet`.

Spark realiza el trabajo que aquí podríamos expresar como un MapReduce. Para esta suma, la idea es asociar cada pedido a una zona y combinar sus unidades. Spark planifica esas tareas a partir de `groupBy()` y `sum()`. HDFS almacena la entrada y el resultado. En esta práctica el cálculo sigue ejecutándose en modo local.

[Configuración de Spark para HDFS](https://spark.apache.org/docs/3.5.7/configuration.html)

In [ ]:
# Sustituye None por tu URI, por ejemplo "hdfs://HOST:PUERTO/user/USUARIO/practica_pedidos".
base_hdfs = None

if base_hdfs:
    entrada_hdfs = base_hdfs.rstrip("/") + "/entrada"
    salida_hdfs = base_hdfs.rstrip("/") + "/resumen"
    pedidos.write.parquet(entrada_hdfs)
    print("Entrada:", entrada_hdfs)
    print("Destino del resumen:", salida_hdfs)

In [ ]:
# Escribe aquí tu solución. Ejecuta antes la preparación con una ruta de HDFS válida.

### Solución

Ejecuta esta celda después de preparar la entrada. Si ya guardaste el resumen en tu solución, vuelve a leerlo para comprobarlo o usa una carpeta nueva antes de repetir la práctica.

In [ ]:
if base_hdfs:
    pedidos_hdfs = spark.read.parquet(entrada_hdfs)
    resumen = pedidos_hdfs.groupBy("zona").agg(F.sum("unidades").alias("total_unidades"))
    resumen.write.parquet(salida_hdfs)

    resultado = spark.read.parquet(salida_hdfs)
    resultado.orderBy("zona").show()
    zona_3 = resultado.filter(F.col("zona") == "zona_3").first()
    assert zona_3["total_unidades"] == total_esperado

In [ ]:
spark.stop()

## Navegación

[Volver al README](../README.md) · [Siguiente: EDA y estadística con PySpark](08_data_preparation_and_analysis.ipynb)